In [1]:
from pathlib import Path
import re
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading"
)

tabular_path = (
    PROJECT_DIR / "ml/grading/data/processed/cleaned_grading_dataset.xlsx"
)
manifest_path = (
    PROJECT_DIR / "ml/grading/image/data/processed/clean_image_manifest.csv"
)

tabular_df = pd.read_excel(tabular_path)
image_df = pd.read_csv(manifest_path)

def normalize_tissue_id(value):
    if pd.isna(value):
        return None

    value = re.sub(r"\s+", " ", str(value)).strip()
    value = re.sub(r"^NEB-\s*", "NEB-", value, flags=re.IGNORECASE)
    return value.casefold()

tabular_df["Match_ID"] = tabular_df["Tissue_ID"].map(normalize_tissue_id)
image_df["Match_ID"] = image_df["Tissue_ID"].map(normalize_tissue_id)

print("Tabular rows:", len(tabular_df))
print("Image manifest rows:", len(image_df))

print("\nDuplicate normalized IDs")
print("Tabular:", tabular_df["Match_ID"].duplicated().sum())
print("Images:", image_df["Match_ID"].duplicated().sum())

tabular_ids = set(tabular_df["Match_ID"].dropna())
image_ids = set(image_df["Match_ID"].dropna())

print("\nMatching audit")
print("Unique tabular IDs:", len(tabular_ids))
print("Unique image IDs:", len(image_ids))
print("IDs in both:", len(tabular_ids & image_ids))
print("Tabular IDs without an image:", len(tabular_ids - image_ids))
print("Image IDs without a tabular record:", len(image_ids - tabular_ids))

print("\nExample tabular IDs without images:")
print(sorted(tabular_ids - image_ids)[:10])

print("\nExample image IDs without tabular records:")
print(sorted(image_ids - tabular_ids)[:10])

Tabular rows: 1054
Image manifest rows: 1024

Duplicate normalized IDs
Tabular: 0
Images: 0

Matching audit
Unique tabular IDs: 1054
Unique image IDs: 1024
IDs in both: 1024
Tabular IDs without an image: 30
Image IDs without a tabular record: 0

Example tabular IDs without images:
['12346 r (year/month unknown)', 'neb-2020-09-12672 r', 'neb-2020-09-12683 l', 'neb-2020-10-12718 l', 'neb-2020-10-12726 r', 'neb-2020-10-12733 l', 'neb-2020-11-12807 l', 'neb-2020-12-12848 r', 'neb-2021-02-12975 r', 'neb-2022-09-14219 r']

Example image IDs without tabular records:
[]


In [3]:
# Keep only the ID and grade columns needed for this audit.
tabular_grades = tabular_df[
    ["Match_ID", "Tissue_Grade_Clean"]
].copy()

image_grades = image_df[
    ["Match_ID", "Tissue_Grade_Clean"]
].copy()

# Prefix the grade columns so both values remain visible after merging.
tabular_grades = tabular_grades.rename(
    columns={"Tissue_Grade_Clean": "Tabular_Grade"}
)
image_grades = image_grades.rename(
    columns={"Tissue_Grade_Clean": "Image_Manifest_Grade"}
)

grade_audit = tabular_grades.merge(
    image_grades,
    on="Match_ID",
    how="inner",
    validate="one_to_one"
)

grade_audit["Grade_Match"] = (
    grade_audit["Tabular_Grade"].fillna("<MISSING>").astype(str)
    == grade_audit["Image_Manifest_Grade"].fillna("<MISSING>").astype(str)
)

print("Matched records:", len(grade_audit))
print("Grade agreements:", int(grade_audit["Grade_Match"].sum()))
print("Grade disagreements:", int((~grade_audit["Grade_Match"]).sum()))

print("\nGrade disagreements or missing grades:")
print(
    grade_audit.loc[
        ~grade_audit["Grade_Match"]
    ].head(20)
)

print("\nTabular grade distribution among matched records:")
print(grade_audit["Tabular_Grade"].value_counts(dropna=False))

print("\nImage manifest grade distribution:")
print(grade_audit["Image_Manifest_Grade"].value_counts(dropna=False))

Matched records: 1024
Grade agreements: 1024
Grade disagreements: 0

Grade disagreements or missing grades:
Empty DataFrame
Columns: [Match_ID, Tabular_Grade, Image_Manifest_Grade, Grade_Match]
Index: []

Tabular grade distribution among matched records:
Tabular_Grade
A+     530
A      314
B      165
NaN     15
Name: count, dtype: int64

Image manifest grade distribution:
Image_Manifest_Grade
A+     530
A      314
B      165
NaN     15
Name: count, dtype: int64


In [4]:
conflicting_ids = {
    normalize_tissue_id("NEB-2024-08-16259 R"),
    normalize_tissue_id("NEB-2025-09-17471 R"),
}

print("Conflicting IDs present in clean manifest:")
print(
    image_df.loc[
        image_df["Match_ID"].isin(conflicting_ids),
        ["Tissue_ID", "Match_ID", "Tissue_Grade_Clean"]
    ]
)

labelled_cohort = grade_audit.dropna(
    subset=["Tabular_Grade", "Image_Manifest_Grade"]
).copy()

labelled_cohort = labelled_cohort[
    ~labelled_cohort["Match_ID"].isin(conflicting_ids)
]

print("\nLabelled matched cohort:", len(labelled_cohort))
print("\nGrade counts:")
print(labelled_cohort["Tabular_Grade"].value_counts())

Conflicting IDs present in clean manifest:
Empty DataFrame
Columns: [Tissue_ID, Match_ID, Tissue_Grade_Clean]
Index: []

Labelled matched cohort: 1009

Grade counts:
Tabular_Grade
A+    530
A     314
B     165
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Start from the confirmed matched and labelled cohort.
split_df = labelled_cohort[
    ["Match_ID", "Tabular_Grade"]
].copy()

split_df = split_df.rename(
    columns={
        "Match_ID": "Normalized_ID",
        "Tabular_Grade": "Tissue_Grade_Clean",
    }
).reset_index(drop=True)

# 70% training, 30% temporary holdout.
train_df, temp_df = train_test_split(
    split_df,
    test_size=0.30,
    stratify=split_df["Tissue_Grade_Clean"],
    random_state=RANDOM_STATE,
)

# Divide the temporary set equally into validation and test.
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Tissue_Grade_Clean"],
    random_state=RANDOM_STATE,
)

train_df = train_df.assign(Split="train")
val_df = val_df.assign(Split="validation")
test_df = test_df.assign(Split="test")

shared_splits = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True,
)

print("Total:", len(shared_splits))
print("\nSplit sizes:")
print(shared_splits["Split"].value_counts())

print("\nGrade counts by split:")
print(pd.crosstab(
    shared_splits["Split"],
    shared_splits["Tissue_Grade_Clean"],
))

print("\nDuplicate IDs:", shared_splits["Normalized_ID"].duplicated().sum())

Total: 1009

Split sizes:
Split
train         706
test          152
validation    151
Name: count, dtype: int64

Grade counts by split:
Tissue_Grade_Clean    A   A+    B
Split                            
test                 47   80   25
train               220  371  115
validation           47   79   25

Duplicate IDs: 0


In [6]:
from pathlib import Path

SPLIT_DIR = (
    PROJECT_DIR / "ml/grading/image/data/processed/splits"
)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = SPLIT_DIR / "multimodal_shared_split.csv"

shared_splits.to_csv(SPLIT_PATH, index=False)

print("Saved shared split manifest to:")
print(SPLIT_PATH)

print("\nSaved columns:")
print(shared_splits.columns.tolist())

print("\nSaved rows:", len(pd.read_csv(SPLIT_PATH)))

Saved shared split manifest to:
C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\image\data\processed\splits\multimodal_shared_split.csv

Saved columns:
['Normalized_ID', 'Tissue_Grade_Clean', 'Split']

Saved rows: 1009


In [7]:
# Load the saved shared split.
split_path = (
    PROJECT_DIR
    / "ml/grading/image/data/processed/splits/multimodal_shared_split.csv"
)
shared_splits = pd.read_csv(split_path)

# Join split assignments to the original tabular records.
tabular_join = tabular_df.copy()
tabular_join["Match_ID"] = tabular_join["Tissue_ID"].map(
    normalize_tissue_id
)

tabular_join = tabular_join.merge(
    shared_splits,
    left_on="Match_ID",
    right_on="Normalized_ID",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_split"),
)

# Attach the selected image path from the clean manifest.
image_paths = image_df[["Match_ID", "Selected_Image"]].copy()

multimodal_df = tabular_join.merge(
    image_paths,
    on="Match_ID",
    how="inner",
    validate="one_to_one",
)

print("Multimodal rows:", len(multimodal_df))
print("Unique tissue IDs:", multimodal_df["Match_ID"].nunique())
print("Missing image paths:", multimodal_df["Selected_Image"].isna().sum())
print("Missing grades:", multimodal_df["Tissue_Grade_Clean"].isna().sum())

print("\nSplit distribution:")
print(multimodal_df["Split"].value_counts())

print("\nGrade distribution:")
print(multimodal_df["Tissue_Grade_Clean"].value_counts())

print("\nColumns needed for the next stage:")
print([
    "Tissue_ID",
    "Tissue_Grade_Clean",
    "Split",
    "Selected_Image",
])

Multimodal rows: 1009
Unique tissue IDs: 1009
Missing image paths: 0
Missing grades: 0

Split distribution:
Split
train         706
test          152
validation    151
Name: count, dtype: int64

Grade distribution:
Tissue_Grade_Clean
A+    530
A     314
B     165
Name: count, dtype: int64

Columns needed for the next stage:
['Tissue_ID', 'Tissue_Grade_Clean', 'Split', 'Selected_Image']


In [8]:
import sys

SCRIPTS_DIR = PROJECT_DIR / "ml/grading/scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import modeling_setup as ms

print("Target:", ms.TARGET)
print("Number of existing features:", len(ms.ALL_FEATURES))

print("\nExisting tabular features:")
for feature in ms.ALL_FEATURES:
    print("-", feature)

missing_features = [
    feature for feature in ms.ALL_FEATURES
    if feature not in multimodal_df.columns
]

print("\nFeatures missing from assembled dataset:", missing_features)

Target: Tissue_Grade_Clean
Number of existing features: 24

Existing tabular features:
- ECD_CD_cells_per_mm2
- CV
- HEX_percent
- SD
- Total_Cell_Count
- Total_Area_um2
- Stroma_Is_Clear
- Stroma_Is_Compact
- Stroma_Has_Arcus
- Stroma_Has_Edema
- Stroma_Has_Scar
- Epi_Has_Arcus
- Epi_Has_Defect
- Epi_Has_Exposure
- Epi_Has_Haze
- Endo_Has_Stress_Lines
- Endo_Has_RBC
- Endo_Has_Cell_Dropout
- Descemet_Has_Folds
- Descemet_Fold_Severity
- Descemet_Fold_Relative_Number
- Endo_Stress_Line_Location
- Stroma_Arcus_Severity
- Epi_Exposure_Severity

Features missing from assembled dataset: []


In [10]:
feature_audit = pd.DataFrame({
    "Feature": ms.ALL_FEATURES,
    "Data_Type": [
        multimodal_df[col].dtype
        for col in ms.ALL_FEATURES
    ],
    "Missing_Count": [
        multimodal_df[col].isna().sum()
        for col in ms.ALL_FEATURES
    ],
    "Missing_Percent": [
        round(multimodal_df[col].isna().mean() * 100, 2)
        for col in ms.ALL_FEATURES
    ],
    "Unique_Values": [
        multimodal_df[col].nunique(dropna=True)
        for col in ms.ALL_FEATURES
    ],
})

print(
    feature_audit.sort_values(
        "Missing_Percent",
        ascending=False
    )
)

print("Total feature cells:", len(multimodal_df) * len(ms.ALL_FEATURES))
print(
    "Missing feature cells:",
    int(multimodal_df[ms.ALL_FEATURES].isna().sum().sum())
)

                          Feature Data_Type  Missing_Count  Missing_Percent  \
23          Epi_Exposure_Severity       str            712            70.56   
22          Stroma_Arcus_Severity       str            551            54.61   
21      Endo_Stress_Line_Location       str            523            51.83   
19         Descemet_Fold_Severity       str            498            49.36   
18             Descemet_Has_Folds   float64            498            49.36   
20  Descemet_Fold_Relative_Number       str            498            49.36   
16                   Endo_Has_RBC   float64            468            46.38   
15          Endo_Has_Stress_Lines   float64            468            46.38   
17          Endo_Has_Cell_Dropout   float64            468            46.38   
13               Epi_Has_Exposure   float64            459            45.49   
11                  Epi_Has_Arcus   float64            459            45.49   
12                 Epi_Has_Defect   float64         

In [11]:
# Use the existing preprocessing pipeline.
from sklearn.base import clone

train_mask = multimodal_df["Split"] == "train"
val_mask = multimodal_df["Split"] == "validation"
test_mask = multimodal_df["Split"] == "test"

X_train_tab = multimodal_df.loc[train_mask, ms.ALL_FEATURES]
X_val_tab = multimodal_df.loc[val_mask, ms.ALL_FEATURES]
X_test_tab = multimodal_df.loc[test_mask, ms.ALL_FEATURES]

y_train = multimodal_df.loc[train_mask, "Tissue_Grade_Clean"]
y_val = multimodal_df.loc[val_mask, "Tissue_Grade_Clean"]
y_test = multimodal_df.loc[test_mask, "Tissue_Grade_Clean"]

# Fit preprocessing on training data only.
tabular_preprocessor = ms.make_preprocessor()

X_train_tab_processed = tabular_preprocessor.fit_transform(X_train_tab)
X_val_tab_processed = tabular_preprocessor.transform(X_val_tab)
X_test_tab_processed = tabular_preprocessor.transform(X_test_tab)

print("Processed tabular shapes:")
print("Train:", X_train_tab_processed.shape)
print("Validation:", X_val_tab_processed.shape)
print("Test:", X_test_tab_processed.shape)

print("\nSample counts:", len(y_train), len(y_val), len(y_test))
print("Transformed feature count:", X_train_tab_processed.shape[1])

Processed tabular shapes:
Train: (706, 68)
Validation: (151, 68)
Test: (152, 68)

Sample counts: 706 151 152
Transformed feature count: 68


In [12]:
import torch
import torch.nn as nn

class MultimodalCorneaModel(nn.Module):
    def __init__(self, tabular_dim=68, num_classes=3):
        super().__init__()

        # Image branch: extract visual features from grayscale images.
        self.image_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

        # Tabular branch: learn a compact representation of 68 features.
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
        )

        # Fusion: concatenate both learned representations.
        self.classifier = nn.Sequential(
            nn.Linear(64 + 32, 48),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(48, num_classes),
        )

    def forward(self, images, tabular):
        image_features = self.image_encoder(images)
        tabular_features = self.tabular_encoder(tabular)

        combined = torch.cat(
            [image_features, tabular_features],
            dim=1,
        )

        return self.classifier(combined)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

fusion_model = MultimodalCorneaModel(
    tabular_dim=X_train_tab_processed.shape[1],
    num_classes=3,
).to(device)

print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in fusion_model.parameters()
    if p.requires_grad
))

Device: cuda
Trainable parameters: 34819


In [13]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 224
CLASS_NAMES = ["A", "A+", "B"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

class MultimodalDataset(Dataset):
    def __init__(self, rows, tabular_features):
        self.rows = rows.reset_index(drop=True).copy()
        self.tabular = np.asarray(
            tabular_features, dtype=np.float32
        )

        assert len(self.rows) == len(self.tabular)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]

        image_path = Path(row["Selected_Image"])

        with Image.open(image_path) as img:
            img = img.convert("L")
            img.thumbnail(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS,
            )

            canvas = Image.new(
                "L", (IMAGE_SIZE, IMAGE_SIZE), color=0
            )
            left = (IMAGE_SIZE - img.width) // 2
            top = (IMAGE_SIZE - img.height) // 2
            canvas.paste(img, (left, top))

            image_array = (
                np.asarray(canvas, dtype=np.float32) / 255.0
            )

        image_tensor = torch.from_numpy(
            image_array[None, :, :].copy()
        )

        tabular_tensor = torch.from_numpy(
            self.tabular[idx].copy()
        )

        label = CLASS_TO_IDX[row["Tissue_Grade_Clean"]]

        return image_tensor, tabular_tensor, label


train_rows = multimodal_df.loc[train_mask].reset_index(drop=True)
val_rows = multimodal_df.loc[val_mask].reset_index(drop=True)
test_rows = multimodal_df.loc[test_mask].reset_index(drop=True)

train_dataset = MultimodalDataset(
    train_rows, X_train_tab_processed
)
val_dataset = MultimodalDataset(
    val_rows, X_val_tab_processed
)
test_dataset = MultimodalDataset(
    test_rows, X_test_tab_processed
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

# Sanity-check one batch.
images, tabular, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Tabular batch:", tabular.shape)
print("Label batch:", labels.shape)
print("Class mapping:", CLASS_TO_IDX)
print("Image values:", float(images.min()), "to", float(images.max()))

Image batch: torch.Size([16, 1, 224, 224])
Tabular batch: torch.Size([16, 68])
Label batch: torch.Size([16])
Class mapping: {'A': 0, 'A+': 1, 'B': 2}
Image values: 0.0 to 1.0


In [14]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

MAX_EPOCHS = 30
PATIENCE = 5
best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
best_state = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    # Training
    fusion_model.train()
    train_loss_sum = 0.0
    train_true, train_pred = [], []

    for images, tabular, labels in train_loader:
        images = images.to(device, non_blocking=True)
        tabular = tabular.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = fusion_model(images, tabular)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_true.extend(labels.cpu().numpy())
        train_pred.extend(logits.argmax(dim=1).cpu().numpy())

    train_loss = train_loss_sum / len(train_dataset)
    train_f1 = f1_score(
        train_true, train_pred, average="macro", zero_division=0
    )

    # Validation
    fusion_model.eval()
    val_loss_sum = 0.0
    val_true, val_pred = [], []

    with torch.no_grad():
        for images, tabular, labels in val_loader:
            images = images.to(device, non_blocking=True)
            tabular = tabular.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = fusion_model(images, tabular)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_true.extend(labels.cpu().numpy())
            val_pred.extend(logits.argmax(dim=1).cpu().numpy())

    val_loss = val_loss_sum / len(val_dataset)
    val_f1 = f1_score(
        val_true, val_pred, average="macro", zero_division=0
    )
    val_acc = accuracy_score(val_true, val_pred)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss {train_loss:.4f} | "
        f"Train macro F1 {train_f1:.4f} | "
        f"Val loss {val_loss:.4f} | "
        f"Val macro F1 {val_f1:.4f} | "
        f"Val accuracy {val_acc:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(fusion_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# Restore the checkpoint selected by validation macro F1.
if best_state is None:
    raise RuntimeError("No model checkpoint was selected.")

fusion_model.load_state_dict(best_state)

history_df = pd.DataFrame(history)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation macro F1: {best_val_f1:.4f}")

Epoch 01 | Train loss 1.0071 | Train macro F1 0.2831 | Val loss 0.9548 | Val macro F1 0.2290 | Val accuracy 0.5232
Epoch 02 | Train loss 0.8344 | Train macro F1 0.4628 | Val loss 0.7324 | Val macro F1 0.6750 | Val accuracy 0.7351
Epoch 03 | Train loss 0.6409 | Train macro F1 0.6667 | Val loss 0.6798 | Val macro F1 0.7224 | Val accuracy 0.7682
Epoch 04 | Train loss 0.5855 | Train macro F1 0.7568 | Val loss 0.6534 | Val macro F1 0.6961 | Val accuracy 0.7483
Epoch 05 | Train loss 0.5345 | Train macro F1 0.7923 | Val loss 0.6290 | Val macro F1 0.7347 | Val accuracy 0.7748
Epoch 06 | Train loss 0.5044 | Train macro F1 0.7729 | Val loss 0.5980 | Val macro F1 0.7343 | Val accuracy 0.7616
Epoch 07 | Train loss 0.4731 | Train macro F1 0.8173 | Val loss 0.7300 | Val macro F1 0.6724 | Val accuracy 0.7550
Epoch 08 | Train loss 0.4687 | Train macro F1 0.8033 | Val loss 0.7104 | Val macro F1 0.6905 | Val accuracy 0.7550
Epoch 09 | Train loss 0.4409 | Train macro F1 0.8325 | Val loss 0.6178 | Val mac

In [15]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import pandas as pd
import numpy as np
import torch

fusion_model.eval()
test_true, test_pred = [], []

with torch.no_grad():
    for images, tabular, labels in test_loader:
        images = images.to(device, non_blocking=True)
        tabular = tabular.to(device, non_blocking=True)

        logits = fusion_model(images, tabular)
        predictions = logits.argmax(dim=1)

        test_true.extend(labels.numpy())
        test_pred.extend(predictions.cpu().numpy())

print("MULTIMODAL TEST RESULTS")
print("=" * 40)
print("Test samples:", len(test_true))
print("Accuracy:", round(accuracy_score(test_true, test_pred), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(test_true, test_pred), 4)
)
print(
    "Macro F1:",
    round(f1_score(test_true, test_pred, average="macro"), 4)
)

print("\nPer-grade classification report:")
print(classification_report(
    test_true,
    test_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    confusion_matrix(test_true, test_pred, labels=[0, 1, 2]),
    index=[f"Actual {c}" for c in CLASS_NAMES],
    columns=[f"Predicted {c}" for c in CLASS_NAMES],
))

MULTIMODAL TEST RESULTS
Test samples: 152
Accuracy: 0.8289
Balanced accuracy: 0.8086
Macro F1: 0.8179

Per-grade classification report:
              precision    recall  f1-score   support

           A     0.8485    0.5957    0.7000        47
          A+     0.8085    0.9500    0.8736        80
           B     0.8800    0.8800    0.8800        25

    accuracy                         0.8289       152
   macro avg     0.8457    0.8086    0.8179       152
weighted avg     0.8326    0.8289    0.8210       152

Confusion matrix (rows = actual, columns = predicted):
           Predicted A  Predicted A+  Predicted B
Actual A            28            16            3
Actual A+            4            76            0
Actual B             1             2           22
